# Video Q&A System with Gemini Pro Vision and ChromaDB

This notebook demonstrates how to:
1. Segment long videos into shorter video clips
2. Use Google Gemini Pro Vision to analyze video segments
3. Generate embeddings using multi2vec
4. Store segments in ChromaDB for question answering

## Setup and Imports

In [21]:
import os
import subprocess
import cv2
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from PIL import Image
import google.generativeai as genai
import chromadb
from chromadb.config import Settings
from tqdm import tqdm
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure Gemini API
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    raise ValueError("Please set GOOGLE_API_KEY in .env file")

genai.configure(api_key=GOOGLE_API_KEY)

## 1. Video Segmentation

Split long videos into shorter video segments

In [22]:
class VideoSegmenter:
    """Split long videos into shorter video segments (with audio) using FFmpeg"""
    
    def __init__(self, video_path: str, output_dir: str = "video_segments"):
        self.video_path = video_path
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
    def segment_video(self, segment_duration: float = 30.0) -> List[Dict]:
        """
        Split video into segments of specified duration, preserving audio.
        Uses FFmpeg to copy both video and audio streams without re-encoding.
        
        Args:
            segment_duration: Duration of each segment in seconds
            
        Returns:
            List of dicts containing segment info (path, start_time, end_time, duration)
        """
        # Use OpenCV only to read video metadata
        cap = cv2.VideoCapture(self.video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        
        print(f"Video: {self.video_path}")
        print(f"FPS: {fps}, Duration: {duration:.2f}s, Resolution: {width}x{height}")
        print(f"Segmenting into {segment_duration}s clips...")
        
        segments_info = []
        segment_index = 0
        current_time = 0
        
        while current_time < duration:
            start_time = current_time
            end_time = min(current_time + segment_duration, duration)
            
            # Create output filename
            segment_filename = f"segment_{segment_index:04d}_t{start_time:.1f}-{end_time:.1f}s.mp4"
            segment_path = self.output_dir / segment_filename
            
            # Use FFmpeg to copy video+audio streams without re-encoding
            cmd = [
                "ffmpeg", "-y",
                "-ss", str(start_time),
                "-i", self.video_path,
                "-t", str(end_time - start_time),
                "-c", "copy",
                "-map", "0:v?",
                "-map", "0:a?",
                "-avoid_negative_ts", "make_zero",
                str(segment_path)
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(f"FFmpeg failed for segment {segment_index}:\n{result.stderr}")
            
            segments_info.append({
                'path': str(segment_path),
                'start_time': start_time,
                'end_time': end_time,
                'duration': end_time - start_time,
                'segment_index': segment_index
            })
            
            print(f"Created segment {segment_index}: {start_time:.1f}s - {end_time:.1f}s")
            
            segment_index += 1
            current_time = end_time
        
        print(f"\nCreated {segment_index} video segments in {self.output_dir}")
        return segments_info

## 2. Gemini Pro Vision Analysis

Use Gemini to analyze and describe video segments

In [23]:
class GeminiAnalyzer:
    """Analyze video segments using Google Gemini Pro Vision"""
    
    def __init__(self, model_name: str = "gemini-2.5-flash"):
        self.model = genai.GenerativeModel(model_name)
        
    def analyze_video_segment(self, video_path: str, prompt: str = None) -> str:
        """
        Analyze a video segment with Gemini Vision
        
        Args:
            video_path: Path to the video file
            prompt: Custom prompt for analysis
            
        Returns:
            Description of the video segment
        """
        if prompt is None:
            prompt = """Analyze this video segment in detail. Provide:
            1. Main subjects, objects, and people
            2. Actions, activities, and events happening
            3. Setting, environment, and location
            4. Any dialogue, text, or audio elements (if visible)
            5. Overall narrative, context, and mood
            6. Key moments or transitions
            
            Be comprehensive but concise."""
        
        # Upload video file to Gemini
        print(f"Uploading {video_path}...")
        video_file = genai.upload_file(path=video_path)
        
        # Wait for video to be processed
        import time
        while video_file.state.name == "PROCESSING":
            time.sleep(2)
            video_file = genai.get_file(video_file.name)
        
        if video_file.state.name == "FAILED":
            raise ValueError(f"Video processing failed for {video_path}")
        
        # Generate content from video
        response = self.model.generate_content([video_file, prompt])
        
        # Clean up uploaded file
        genai.delete_file(video_file.name)
        
        return response.text
    
    def analyze_segments_batch(self, segments_info: List[Dict]) -> List[Dict]:
        """
        Analyze multiple video segments with descriptions
        
        Args:
            segments_info: List of segment information dicts
            
        Returns:
            Updated segments_info with 'description' field added
        """
        print(f"Analyzing {len(segments_info)} video segments with Gemini Pro Vision...")
        
        for i, segment_info in enumerate(tqdm(segments_info, desc="Analyzing segments")):
            try:
                print(f"\nProcessing segment {i+1}/{len(segments_info)}...")
                description = self.analyze_video_segment(segment_info['path'])
                segment_info['description'] = description
                print(f"\nsegment description: {description}")
            except Exception as e:
                print(f"\nError analyzing segment {i}: {e}")
                segment_info['description'] = f"Error: {str(e)}"
        
        return segments_info

## 3. ChromaDB Integration

Store video segments with embeddings in ChromaDB

In [24]:
class VideoChromaDB:
    """Store and query video segments in ChromaDB"""
    
    def __init__(self, collection_name: str = "video_segments", persist_directory: str = "./chroma_db"):
        self.client = chromadb.PersistentClient(path=persist_directory)
        
        # Create or get collection with multi-modal embedding
        # ChromaDB will use default embedding function (all-MiniLM-L6-v2)
        # For multi2vec, you would need to implement custom embedding function
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "Video segments with descriptions"}
        )
        
    def add_segments(self, segments_info: List[Dict], video_name: str):
        """
        Add analyzed video segments to ChromaDB
        
        Args:
            segments_info: List of segment dicts with descriptions
            video_name: Name of the source video
        """
        documents = []
        metadatas = []
        ids = []
        
        for segment in segments_info:
            if 'description' not in segment:
                continue
                
            doc_id = f"{video_name}_segment_{segment['segment_index']}"
            
            documents.append(segment['description'])
            metadatas.append({
                'video_name': video_name,
                'start_time': segment['start_time'],
                'end_time': segment['end_time'],
                'duration': segment['duration'],
                'segment_path': segment['path']
            })
            ids.append(doc_id)
        
        print(f"Adding {len(documents)} video segments to ChromaDB...")
        self.collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        print(f"Successfully added {len(documents)} segments to collection '{self.collection.name}'")
    
    def query(self, question: str, n_results: int = 5) -> Dict:
        """
        Query the video segments
        
        Args:
            question: Natural language question
            n_results: Number of results to return
            
        Returns:
            Query results with relevant segments
        """
        results = self.collection.query(
            query_texts=[question],
            n_results=n_results
        )
        return results
    
    def format_results(self, results: Dict) -> List[Dict]:
        """
        Format query results for display
        
        Args:
            results: Raw ChromaDB query results
            
        Returns:
            Formatted list of result dicts
        """
        formatted = []
        
        for i in range(len(results['ids'][0])):
            formatted.append({
                'id': results['ids'][0][i],
                'description': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
        
        return formatted

## 4. Complete Pipeline

Put it all together

In [25]:
# Configuration
VIDEO_PATH = "downloads/International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage).mp4"  # Update this path
SEGMENT_DURATION = 30.0  # Duration of each video segment in seconds
VIDEO_NAME = Path(VIDEO_PATH).stem

print(f"Processing video: {VIDEO_NAME}")
print(f"Segment duration: {SEGMENT_DURATION}s")

Processing video: International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage)
Segment duration: 30.0s


In [26]:
# Step 1: Segment video into shorter clips
segmenter = VideoSegmenter(VIDEO_PATH, output_dir=f"video_segments_{VIDEO_NAME}")
segments_info = segmenter.segment_video(segment_duration=SEGMENT_DURATION)

Video: downloads/International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage).mp4
FPS: 29.97002997002997, Duration: 200.17s, Resolution: 3840x2160
Segmenting into 30.0s clips...
Created segment 0: 0.0s - 30.0s
Created segment 1: 30.0s - 60.0s
Created segment 2: 60.0s - 90.0s
Created segment 3: 90.0s - 120.0s
Created segment 4: 120.0s - 150.0s
Created segment 5: 150.0s - 180.0s
Created segment 6: 180.0s - 200.2s

Created 7 video segments in video_segments_International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage)


In [27]:
# Step 2: Analyze video segments with Gemini Pro Vision
analyzer = GeminiAnalyzer()
segments_info = analyzer.analyze_segments_batch(segments_info)

Analyzing 7 video segments with Gemini Pro Vision...


Analyzing segments:   0%|          | 0/7 [00:00<?, ?it/s]


Processing segment 1/7...
Uploading video_segments_International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage)/segment_0000_t0.0-30.0s.mp4...


Analyzing segments:  14%|█▍        | 1/7 [00:46<04:39, 46.65s/it]


segment description: Here's a detailed analysis of the video segment:

1.  **Main subjects, objects, and people:**
    *   **Subjects:** The International Space Station (ISS), astronauts (including Jasmin Moghbeli), Earth.
    *   **Objects:** ISS modules, solar panels, scientific experimental equipment (microgravity freezer, multi-well plates, crystal growth apparatus), exercise equipment (barbell-like device, weighted vest), cameras, laptops, various cables and instruments.
    *   **People:** Multiple diverse astronauts (male and female) are shown engaging in various activities.

2.  **Actions, activities, and events happening:**
    *   **Views:** Exterior shots of the ISS orbiting Earth, interior views looking out through the cupola at Earth, nighttime views of Earth with city lights and aurora.
    *   **Scientific Research:** Astronauts are shown conducting experiments, handling samples (e.g., crystal growth, liquids in multi-well plates), and operating specialized equipment in

...


segment description: Here's a detailed analysis of the video segment:

**1. Main Subjects, Objects, and People:**
*   **People:** Astronauts (at least 7 visible, varying ethnicities), children (one girl, one boy).
*   **Objects:** Scientific experiment equipment, transparent sample bags ("MABL-A Sample Return Bag," "Trash Bag"), microscopic crystalline structures, International Space Station (ISS) modules, solar arrays, robotic arm (Canadarm2), SpaceX Dragon capsule, professional camera with a large lens, a detailed 3D animated model of Earth, industrial equipment with glowing elements, white fabric bag being handled by an astronaut.
*   **Concepts:** Scientific research, space exploration, Earth observation, disease fighting, economic opportunity, inspiration.

**2. Actions, Activities, and Events Happening:**
*   **0:00-0:04:** An astronaut in blue gloves and a white lab coat is meticulously handling scientific experiment equipment, possibly processing samples, within the ISS. The t

Analyzing segments:  86%|████████▌ | 6/7 [07:14<01:21, 81.14s/it]


segment description: Here's a detailed analysis of the video segment:

1.  **Main Subjects, Objects, and People:**
    *   **People:** Several children and young adults (appearing as students/public), multiple NASA astronauts (including Mark Vande Hei and Kayla Barron), and other international astronauts.
    *   **Objects:** The International Space Station (ISS), chili pepper plants, a plant habitat/grow chamber, food/drink pouches, a microphone, various scientific equipment and modules within the ISS.
    *   **Celestial Bodies:** Earth, stars, aurora borealis/australis.

2.  **Actions, Activities, and Events Happening:**
    *   **Inspiration & Collaboration:** Children and young adults are shown speaking, seemingly engaging in a virtual Q&A or motivational segment with astronauts. Astronauts are seen interacting with each other, showing camaraderie.
    *   **Q&A:** A young boy asks a question to the astronauts about their favorite experiment.
    *   **Scientific Experimentation:

Analyzing segments: 100%|██████████| 7/7 [08:08<00:00, 69.81s/it]


segment description: Here's a detailed analysis of the video segment:

1.  **Main subjects, objects, and people:**
    *   **Subjects:** The International Space Station (ISS) itself (both exterior and interior), Earth, the Moon, aurora borealis/australis.
    *   **Objects:** ISS modules, solar panels, various scientific equipment and experiments (e.g., plant growth systems with tomatoes/peas, petri dish-like trays), computers, cameras.
    *   **People:** Several astronauts of diverse genders and ethnicities are shown, performing various activities.

2.  **Actions, activities, and events happening:**
    *   The ISS is seen orbiting Earth, passing over aurora displays.
    *   Astronauts are engaged in scientific research (cultivating plants, handling experimental samples).
    *   They are observing and photographing Earth from the Cupola module.
    *   Astronauts are shown collaborating, interacting, hugging, and posing for a group photo, highlighting teamwork and camaraderie in s

In [28]:
# Step 3: Store segments in ChromaDB
db = VideoChromaDB(collection_name="video_qa_collection")
db.add_segments(segments_info, VIDEO_NAME)

Adding 7 video segments to ChromaDB...
Successfully added 7 segments to collection 'video_qa_collection'


## 5. Query the Video

Ask questions about the video content

In [29]:
# Example queries
questions = [
    #"What activities are happening in the video?",
    #"Are there any people in the video?",
    "What is the setting or location?",
    #"What objects are visible?",
    "How far is ISS from Earth?",
    "What research is carried out in ISS?"
    "How ISS research is used in FIFA World Cup 2026?"
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    
    results = db.query(question, n_results=3)
    formatted_results = db.format_results(results)
    
    for i, result in enumerate(formatted_results, 1):
        print(f"\nResult {i}:")
        print(f"Time Range: {result['metadata']['start_time']:.1f}s - {result['metadata']['end_time']:.1f}s")
        print(f"Segment: {result['metadata']['segment_path']}")
        print(f"Description: {result['description'][:200]}...")
        if result['distance']:
            print(f"Relevance Score: {1 - result['distance']:.3f}")


Question: What is the setting or location?

Result 1:
Time Range: 60.0s - 90.0s
Segment: video_segments_International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage)/segment_0002_t60.0-90.0s.mp4
Description: This video segment is a montage showcasing various aspects of life, work, and the environment related to the International Space Station (ISS) and space exploration.

**1. Main subjects, objects, and ...
Relevance Score: -0.624

Result 2:
Time Range: 30.0s - 60.0s
Segment: video_segments_International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage)/segment_0001_t30.0-60.0s.mp4
Description: Here's a detailed analysis of the video segment:

1.  **Main subjects, objects, and people:**
    *   **People:** Multiple astronauts of various nationalities (male and female) are featured, including...
Relevance Score: -0.627

Result 3:
Time Range: 0.0s - 30.0s
Segment: video_segments_International Space Station： Humanity’s Lab in Space (Narrated by Adam Savag

## 6. Interactive Query Interface

In [30]:
def query_video(question: str, n_results: int = 3):
    """Interactive function to query the video"""
    results = db.query(question, n_results=n_results)
    formatted_results = db.format_results(results)
    
    print(f"\nQuestion: {question}")
    print(f"Found {len(formatted_results)} relevant segments:\n")
    
    for i, result in enumerate(formatted_results, 1):
        print(f"\n--- Segment {i} ---")
        print(f"Time Range: {result['metadata']['start_time']:.1f}s - {result['metadata']['end_time']:.1f}s")
        print(f"Duration: {result['metadata']['duration']:.1f}s")
        print(f"Description: {result['description']}")
        print(f"Segment Path: {result['metadata']['segment_path']}")

# Example usage:
query_video("What is the setting or location?")


Question: What is the setting or location?
Found 3 relevant segments:


--- Segment 1 ---
Time Range: 60.0s - 90.0s
Duration: 30.0s
Description: This video segment is a montage showcasing various aspects of life, work, and the environment related to the International Space Station (ISS) and space exploration.

**1. Main subjects, objects, and people:**
*   **People:** Multiple astronauts (both male and female, of diverse backgrounds) are featured, performing different tasks inside and outside the ISS.
*   **Objects/Subjects:**
    *   Scientific instruments, microscopes, computer screens displaying data, laptops, VR headsets.
    *   Petri dishes or wells with biological samples (seen on the monitor).
    *   The International Space Station (ISS) itself, both interior modules (packed with wires, equipment, personal items) and its exterior (solar arrays, robotic arm labeled "Canada," docking ports).
    *   Astronaut suits (EVA suit and interior flight suits).
    *   Flags of various 

## 7. Statistics and Summary

In [31]:
# Get collection statistics
collection_count = db.collection.count()
print(f"\nCollection Statistics:")
print(f"Total segments stored: {collection_count}")
print(f"Collection name: {db.collection.name}")
print(f"\nVideo processed: {VIDEO_NAME}")
print(f"Total segments created: {len(segments_info)}")
if segments_info:
    print(f"Video duration covered: 0s to {segments_info[-1]['end_time']:.1f}s")


Collection Statistics:
Total segments stored: 7
Collection name: video_qa_collection

Video processed: International Space Station： Humanity’s Lab in Space (Narrated by Adam Savage)
Total segments created: 7
Video duration covered: 0s to 200.2s
